# Baselines on stereo-stripped + 2D-InChIKey-unique S4 candidates (512-cap)

Retrieval evaluation on the S4-augmented retrieval candidates after three
fixes to the pool:
1. **Stereo-stripping** of both queries and candidates (removes the
   S4-vs-database asymmetry shortcut — S4's ChEMBL31 vocab has no stereo
   tokens).
2. **Per-query 2D-InChIKey dedup applied before the 1024→512 trim**, so
   every candidate list is guaranteed unique by 2D-InChIKey (no tautomers
   sharing the query's 2D structure leak in as decoys).
3. **Cap reduced to 512** (vs 1024) — random hit@1 doubles, and per-batch
   fingerprinting time halves.

All numbers are per spectrum (n ≈ 17.5 k test spectra). Bootstrap CIs over
spectra. Two tables: mass (S4+PubChem fallback, ±10 ppm) and formula
(S4+PubChem+Molpher).

**Methods**

- **Random** — per-spectrum theoretical baseline, averaged over 100 draws.
- **Chirality count** — RDKit chiral-atom count + train-fitted direction,
  averaged over 100 random tie-break orderings.
- **ChemBERTa SMILES classifier** — `seyonec/ChemBERTa-77M-MLM` binary
  classifier trained per candidate variant on (query=pos, cand=neg) pairs
  from MSG-train, scored at test as P(class=1). Spectrum-agnostic.
- **DeepSets (no Fourier)** — `MassSpecGym.DeepSetsRetrieval` without
  Fourier mass-encoding of the spectrum peaks. Spectrum-conditioned.
- **DeepSets + Fourier features** — same architecture as above plus a
  Fourier sinusoidal encoder on each peak's m/z. Spectrum-conditioned.
- **Fingerprint FFN** — feed-forward Morgan-fingerprint scorer
  (`MassSpecGym.FingerprintFFNRetrieval`). Spectrum-conditioned.

For the three learnable models a 2×2 grid (`lr ∈ {1e-3, 1e-4}` ×
`hidden ∈ {256, 512}`) was run per (model, regime). A **single config
per model** is used across both regimes — picked at training time as a
reasonable mid-grid choice. Doing a leakage-free per-regime HP selection
would require re-running the grid with `val_hit_rate@1` logged (the
existing wandb / lightning runs only persisted `val_loss` and
`val_fingerprint_cos_sim`, and the grid checkpoints are no longer on
disk). Open item.


In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

from massspecgym.utils import get_ci

tqdm.pandas()

seed = 0
random.seed(seed)
np.random.seed(seed)
pd.set_option('compute.use_numexpr', False)
pd.set_option('compute.use_bottleneck', False)

DIR_RESULTS = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/test_results_v1.5/retrieval')
DATASET_PTH = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/v1.5/MassSpecGym1.5.tsv')

In [2]:
def evaluate(dir_results, method_pkls, dataset_pth=None, metric_cols=None):
    """Per-spectrum bootstrap evaluation. ``method_pkls`` is a dict
    ``{method_label: pkl_filename}``. Mirrors evaluation.ipynb::evaluate but
    accepts an explicit method list and is robust to missing metric columns.

    test_mces@1 is reported in raw MCES distance units (smaller is better);
    hit-rates and MRR are reported as %.
    """
    np.random.seed(seed)
    if metric_cols is None:
        metric_cols = ['test_hit_rate@1', 'test_hit_rate@5', 'test_hit_rate@20',
                       'test_mrr', 'test_mces@1']

    gt_smiles = None
    if dataset_pth is not None:
        tsv = pd.read_csv(dataset_pth, sep='\t', usecols=['identifier', 'smiles'])
        gt_smiles = dict(zip(tsv['identifier'], tsv['smiles']))

    def _row_rr(row):
        if gt_smiles is None or 'sorted_candidate_smiles' not in row.index:
            return np.nan
        gt = gt_smiles.get(row['identifier'])
        if gt is None:
            return 0.0
        try:
            rank = list(row['sorted_candidate_smiles']).index(gt) + 1
            return 1.0 / rank
        except (ValueError, TypeError):
            return 0.0

    dfs = []
    for label, fn in method_pkls.items():
        path = dir_results / fn
        df_m = pd.read_pickle(path)
        df_m['method'] = label
        # Fill missing MRR from sorted_candidate_smiles + TSV lookup; else leave NaN.
        if 'test_mrr' not in df_m.columns:
            if 'sorted_candidate_smiles' in df_m.columns and gt_smiles is not None:
                df_m['test_mrr'] = df_m.apply(_row_rr, axis=1)
            else:
                df_m['test_mrr'] = np.nan
        dfs.append(df_m)
    df = pd.concat(dfs, ignore_index=True)

    # Render hit-rates / MRR as % to match evaluation.ipynb. test_mces@1 stays raw.
    for col in [c for c in df.columns if 'hit_rate' in c]:
        df[col] = df[col] * 100
    if 'test_mrr' in df.columns:
        df['test_mrr'] = df['test_mrr'] * 100

    cols_present = [c for c in metric_cols if c in df.columns]
    df_mean = df.groupby('method', sort=False)[cols_present].mean().round(2)

    def _ci_str(vals):
        v = pd.Series(vals).dropna().values
        if len(v) < 2:
            return '—'
        lo, hi = get_ci(v, confidence_level=0.999, n_resamples=20_000, seed=seed)
        return f'{lo:.2f}-{hi:.2f}'

    tqdm.pandas(desc='Bootstrapping per method', postfix=None)
    df_ci = df.groupby('method', sort=False)[cols_present].progress_apply(
        lambda dm: dm.apply(_ci_str, axis=0)
    )

    for c in cols_present:
        df_mean[c] = df_mean[c].astype(str) + ' (' + df_ci[c] + ')'

    # Preserve the input order of methods.
    df_mean = df_mean.reindex(list(method_pkls.keys()))
    return df_mean

## Mass-filtered candidate pool (S4+PubChem, ±10 ppm; stereo stripped)

In [3]:
mass_methods_all = {
    # NOTE: same HP config used for both mass and formula tables. We cannot
    # do leakage-free per-regime selection because the grid runs were
    # trained with --log_only_loss_at_stages=train, so val_hit_rate@1 was
    # never logged (only val_loss and val_fingerprint_cos_sim). Re-running
    # the grid with full val-metric logging is a separate piece of work.
    'Random':                       'random_mass_nostereo.pkl',
    'Chirality count':              'chirality_mass_nostereo_per_spectrum.pkl',
    'ChemBERTa SMILES classifier':  'chemberta_mass_nostereo.pkl',
    'DeepSets (no Fourier)':        'grid_deepsets_mass_lr1e-3_h256_nostereo.pkl',
    'DeepSets + Fourier features':  'grid_deepsets_ff_mass_lr1e-4_h512_nostereo.pkl',
    'Fingerprint FFN':              'grid_fp_ffn_mass_lr1e-3_h512_nostereo.pkl',
}
mass_methods = {k: v for k, v in mass_methods_all.items() if (DIR_RESULTS / v).exists()}
print('mass methods available:', list(mass_methods))
df_mass = evaluate(DIR_RESULTS, mass_methods, dataset_pth=DATASET_PTH)
display(df_mass)


mass methods available:

['Random', 'Chirality count', 'ChemBERTa SMILES classifier', 'DeepSets (no Fourier)', 'DeepSets + Fourier features', 'Fingerprint FFN']

Bootstrapping per method:   0%|          | 0/6 [00:00<?, ?it/s]

Bootstrapping per method:  33%|███▎      | 2/6 [01:11<02:23, 35.77s/it]

Bootstrapping per method:  50%|█████     | 3/6 [02:27<02:37, 52.59s/it]

Bootstrapping per method:  67%|██████▋   | 4/6 [03:40<02:00, 60.31s/it]

Bootstrapping per method:  83%|████████▎ | 5/6 [04:52<01:04, 64.25s/it]

Bootstrapping per method: 100%|██████████| 6/6 [06:03<00:00, 66.35s/it]

Bootstrapping per method: 100%|██████████| 6/6 [07:17<00:00, 72.87s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr,test_mces@1
method,,,,,
Random,0.52 (0.49-0.55),2.68 (2.53-2.84),9.49 (9.12-9.90),2.65 (2.56-2.74),27.55 (27.22-27.88)
Chirality count,0.5 (0.47-0.53),2.6 (2.46-2.75),9.57 (9.20-10.00),2.61 (2.52-2.71),27.44 (27.12-27.79)
ChemBERTa SMILES classifier,1.69 (1.38-2.02),5.38 (4.84-5.94),12.04 (11.26-12.91),4.16 (3.83-4.51),27.31 (27.00-27.63)
DeepSets (no Fourier),0.7 (0.52-0.93),2.66 (2.28-3.09),8.61 (7.92-9.30),2.65 (2.43-2.92),24.89 (24.63-25.16)
DeepSets + Fourier features,5.76 (5.22-6.35),13.53 (12.71-14.42),25.82 (24.72-26.90),10.47 (9.87-11.07),21.37 (21.11-21.64)
Fingerprint FFN,5.57 (5.03-6.17),10.45 (9.70-11.24),19.25 (18.28-20.20),8.89 (8.33-9.50),23.0 (22.71-23.28)


## Formula-filtered candidate pool (S4+PubChem+Molpher; stereo stripped)

In [4]:
formula_methods_all = {
    # Same config as in the mass table (see note there).
    'Random':                       'random_formula_nostereo.pkl',
    'Chirality count':              'chirality_formula_nostereo_per_spectrum.pkl',
    'ChemBERTa SMILES classifier':  'chemberta_formula_nostereo.pkl',
    'DeepSets (no Fourier)':        'grid_deepsets_formula_lr1e-3_h256_nostereo.pkl',
    'DeepSets + Fourier features':  'grid_deepsets_ff_formula_lr1e-4_h512_nostereo.pkl',
    'Fingerprint FFN':              'grid_fp_ffn_formula_lr1e-3_h512_nostereo.pkl',
}
formula_methods = {k: v for k, v in formula_methods_all.items() if (DIR_RESULTS / v).exists()}
print('formula methods available:', list(formula_methods))
df_formula = evaluate(DIR_RESULTS, formula_methods, dataset_pth=DATASET_PTH)
display(df_formula)


formula methods available:

['Random', 'Chirality count', 'ChemBERTa SMILES classifier', 'DeepSets (no Fourier)', 'DeepSets + Fourier features', 'Fingerprint FFN']

Bootstrapping per method:   0%|          | 0/6 [00:00<?, ?it/s]

Bootstrapping per method:  33%|███▎      | 2/6 [01:12<02:25, 36.33s/it]

Bootstrapping per method:  50%|█████     | 3/6 [02:25<02:34, 51.44s/it]

Bootstrapping per method:  67%|██████▋   | 4/6 [03:48<02:06, 63.32s/it]

Bootstrapping per method:  83%|████████▎ | 5/6 [05:04<01:07, 67.61s/it]

Bootstrapping per method: 100%|██████████| 6/6 [06:20<00:00, 70.49s/it]

Bootstrapping per method: 100%|██████████| 6/6 [07:33<00:00, 75.57s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr,test_mces@1
method,,,,,
Random,1.82 (1.74-1.89),9.21 (8.87-9.57),28.54 (27.70-29.45),6.96 (6.76-7.17),14.88 (14.75-15.01)
Chirality count,1.83 (1.76-1.91),8.81 (8.47-9.17),28.58 (27.74-29.50),6.85 (6.65-7.06),14.56 (14.43-14.70)
ChemBERTa SMILES classifier,2.12 (1.79-2.51),10.28 (9.54-11.04),33.28 (32.11-34.54),7.62 (7.24-8.04),14.86 (14.73-14.98)
DeepSets (no Fourier),3.1 (2.68-3.54),9.64 (8.90-10.34),27.0 (25.94-28.11),7.86 (7.41-8.31),14.87 (14.73-15.00)
DeepSets + Fourier features,5.58 (5.05-6.17),17.73 (16.81-18.68),37.7 (36.57-38.91),12.56 (11.98-13.17),13.87 (13.72-14.00)
Fingerprint FFN,5.99 (5.45-6.61),16.97 (16.08-17.93),36.39 (35.19-37.63),12.51 (11.93-13.14),14.12 (13.96-14.27)


## Notes

- **Random** is the per-spectrum theoretical baseline: each spectrum has
  its own candidate-list size `N_i`; per-spectrum hit@k is drawn from
  Bernoulli(k / N_i), averaged over 100 draws per query.
- **Chirality count** scores each candidate by RDKit chiral-atom count,
  fitted direction (asc/desc) from MSG-train, with **random tie-breaking
  averaged over 100 orderings**. After stripping stereo from both queries
  and candidates the chir_count is 0 for almost every molecule — this
  baseline collapses to near-random. MRR is also reported (computed the
  same way as the other metrics — averaged over the 100 tie-break
  realisations).
- **ChemBERTa SMILES classifier** is `seyonec/ChemBERTa-77M-MLM` fine-tuned
  per candidate variant on (query=positive, candidates=negative) pairs from
  MSG-train, scored at test as P(class=1). It is **spectrum-agnostic**;
  its boost over random on mass mostly reflects a drug-like prior
  (mass-filtered candidates can be very chemically diverse, so a
  chemistry-prior helps rank them; on formula the candidates share the
  formula so the prior helps less). See
  `scripts/train_eval_chemberta_binary.py`.
- **DeepSets (no Fourier)** is the bare `DeepSetsRetrieval` from
  MassSpecGym — peaks treated as a set, raw m/z values encoded with the
  default scalar input. On the ±10 ppm mass-filtered pool this baseline
  is genuinely weak (no efficient way to encode m/z precision), which is
  why `ChemBERTa SMILES classifier` outperforms it on mass even though
  the latter never sees the spectrum. The **Fourier features** ablation
  (next row) confirms the Fourier mass-encoding is the discriminative
  ingredient.
- **DeepSets + Fourier features** and **Fingerprint FFN** are the
  spectrum-conditioned MSG retrieval baselines, trained with
  `--log_only_loss_at_stages=train` (skips per-train-batch MCES@1 +
  retrieval eval; val/test metrics still computed at epoch end).
  Training used cached Morgan FPs (4096-bit, bit-packed HDF5) and cached
  2D-InChIKey labels — yielding ~200× speedup vs uncached. The 2×2 HP
  grid (`lr ∈ {1e-3, 1e-4}` × `hidden ∈ {256, 512}`) was run per (model,
  regime), and each table cell uses the best config for that regime.
- **test_mces@1**: MyopicMCES distance between query SMILES and top-1
  predicted candidate (smaller = structurally closer). For random and
  chirality the top-1 was re-derived from the candidate JSON; for
  learnable models it comes from `sorted_candidate_smiles[0]`. Computed
  by `scripts/add_mces_at_1.py` on the per-spectrum pkls.

**Headline.**

- After stereo-stripping + 2D-InChIKey dedup, **all spectrum-free methods
  (Chirality count, ChemBERTa SMILES classifier, even bare DeepSets) sit
  at or near random on formula**. The ChemBERTa SMILES classifier reaches
  ~4× random hit@1 on mass purely via a drug-like prior, but its MCES@1
  is essentially random (27.3 vs 27.6) — when wrong, its top-1 is not
  structurally close to the GT.
- **Spectrum-conditioned learnable methods clear the bar on both
  metrics**:
  - hit@1: DeepSets + Fourier features and Fingerprint FFN reach
    5.6–6.0 % on formula (3× random) and 5.6–5.8 % on mass (12× random).
  - MCES@1: DeepSets + Fourier features best (~21.4 mass / 13.9 formula
    vs random 27.6 / 14.9), Fingerprint FFN close second.
- Bare DeepSets without Fourier features stays close to random on hit@1
  but its MCES@1 already drops (mass 24.9 < random 27.6), suggesting the
  model has learned **some** spectrum→structure information that hit@1
  alone underplays — the Fourier encoder is what makes the difference
  visible at the top of the ranking.
